In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import timm


from paddy_10_data_loader import load_train_val_data
from shufflenet_v2 import ShuffleNetV2

from kd_utils import student_training_loop, evaluate
from helper_utils import count_params


In [2]:
train_loader, val_loader = load_train_val_data(batch_size=32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## ShuffleNetV2 as a student model with knowledge distillation

In [3]:
densenet_201_teacher = timm.create_model("densenet201", pretrained=False, num_classes=10)
densenet_201_teacher.load_state_dict(torch.load("densenet201_teacher.pth"))

<All keys matched successfully>

In [4]:
shufflenet_v2_student = ShuffleNetV2(n_class=10, model_size="0.90x")

model size is  0.90x


In [5]:
count_params(shufflenet_v2_student)

Total Parameters:         1,191,708
Trainable Parameters:     1,176,232
Non-trainable Parameters: 15,476


In [6]:
shufflenet_v2_student_loss = nn.CrossEntropyLoss()
shufflenet_v2_student_optimizer = optim.Adam(shufflenet_v2_student.parameters(), lr=0.001)

In [7]:
trained_shufflenet_v2_student, shufflenet_v2_student_history = student_training_loop(
   teacher=densenet_201_teacher,
   student=shufflenet_v2_student,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=shufflenet_v2_student_optimizer,
    temperature=4.0,
    alpha=0.8,
    num_epochs=15,
    device=device,
    scheduler=None,
    save_path= "shufflenet_v2_student.pth",
)

Overall Progress:   0%|          | 0/4905 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 36.1889%, Train Hard Loss: 1.9527, Train Soft Loss: 1.0137, Train Distill Loss: 4.8060 | Val Hard Loss: 1.6502, Val Soft Loss: 0.7854, Val Distill Loss: 7.1081, Val Acc: 46.8330%
Epoch 2/15 - Train Acc: 50.7750%, Train Hard Loss: 1.5753, Train Soft Loss: 0.7838, Train Distill Loss: 3.7686 | Val Hard Loss: 1.5214, Val Soft Loss: 0.6266, Val Distill Loss: 5.7734, Val Acc: 53.9827%
Epoch 3/15 - Train Acc: 63.2104%, Train Hard Loss: 1.2596, Train Soft Loss: 0.6052, Train Distill Loss: 2.9445 | Val Hard Loss: 1.2756, Val Soft Loss: 0.4785, Val Distill Loss: 4.4656, Val Acc: 65.3551%
Epoch 4/15 - Train Acc: 72.9545%, Train Hard Loss: 0.9588, Train Soft Loss: 0.4510, Train Distill Loss: 2.2101 | Val Hard Loss: 1.1253, Val Soft Loss: 0.3980, Val Distill Loss: 3.7469, Val Acc: 71.4971%
Epoch 5/15 - Train Acc: 79.9111%, Train Hard Loss: 0.6915, Train Soft Loss: 0.3434, Train Distill Loss: 1.6522 | Val Hard Loss: 0.8829, Val Soft Loss: 0.3169, Val Distill Loss: 2.9765, Val